In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers_opti import*
from helpers_setup import *
from helpers_output import *
from rjd_base import *
from tqdm import tqdm
import time
from scipy.sparse.linalg import eigsh

## Tests

In [ ]:
ALL_KEYS = ["sG", "subspace invariance", "NMI", "AMI", "nb iter", "time/iter (avg)"]
SCORE_KEYS = ["sG", "subspace invariance", "NMI", "AMI"]
EXTRA_KEYS = ["nb iter", "time/iter (avg)"]

df_mean   = pd.DataFrame(columns=ALL_KEYS)
df_std    = pd.DataFrame(columns=ALL_KEYS)
df_ci_low = pd.DataFrame(columns=ALL_KEYS)
df_ci_upp = pd.DataFrame(columns=ALL_KEYS)

def make_result(scores, nb_iter, time):
    return dict(zip(ALL_KEYS, [*scores, nb_iter, time]))

def add_avgresults_to_df(results_dict, method, confidence=0.95):
    """
    results_dict : list of per-run result dicts, each with keys ALL_KEYS
    Fills df_mean, df_std, df_ci_low, df_ci_upp in place.
    """

    dfs    = [df_mean, df_std, df_ci_low, df_ci_upp]
    rows   = [[], [], [], []]
    for key in ALL_KEYS:
        values = [r[key] for r in results_dict]
        for row, stat in zip(rows, compute_stats(values, confidence)):
            row.append(stat)

    for df, row in zip(dfs, rows):
        df.loc[method] = row

In [ ]:
example = "synthetic1" # change to "caltech" or "weighted_sbm" when needed
nb_repet = 10

if example == "weighted_sbm":
    L, k, Y_true = example_weighted_sbm()
elif example == "caltech":
    L, k, Y_true = load_example_caltech()
else:
    L, k, Y_true = load_example_1(0.8, 0.3, 50, 0, 0.05)
n = L[0].shape[0]
step_gd = 0.1
alpha = 10
tol_conv = 1e-6 # tolerance criteria for gradient descent method (norm of difference of consecutive iteration)
nb_iter = 1000 # max iteration for gradient descent

In [ ]:
# average method
# performing multiples times this method only changes the time since there is no randomness in it
results_avg = []
for _ in tqdm(range(nb_repet)):
    t0 = time.time()
    avg_L = np.mean(L, axis=0)
    _, X_avg = eigsh(avg_L, k=k, which='SA')  # compute the bottom-k eigenvalues and corresponding eigenvectors
    # eigvals, eigvecs = np.linalg.eigh(avg_L)
    # idx = np.argsort(eigvals)[:k] # indices of the k smallest eigenvalues (probably not needed to sort before)
    # X_avg = eigvecs[:, idx]
    t1 = time.time()
    results_avg.append(make_result(compute_scores(X_avg, L, k, Y_true), 0, t1-t0))
add_avgresults_to_df(results_avg, "Average method")

In [ ]:
results_2 = []
results_100 = []
results_1000 = []

for _ in tqdm(range(nb_repet)):
    X_rjd2, t_rjd2 = rjd_base_timed(L, k, 2)
    X_rjd100, t_rjd100 = rjd_base_timed(L, k, 100)
    X_rjd1000, t_rjd1000 = rjd_base_timed(L,k,1000)
    results_2.append(make_result(compute_scores(X_rjd2, L, k, Y_true), 2, t_rjd2))
    results_100.append(make_result(compute_scores(X_rjd100, L, k, Y_true), 100, t_rjd100))
    results_1000.append(make_result(compute_scores(X_rjd1000, L, k, Y_true), 1000, t_rjd1000))
    
add_avgresults_to_df(results_2, "RJD-BASE (T=2)")
add_avgresults_to_df(results_100, "RJD-BASE (T=100)")
add_avgresults_to_df(results_1000, "RJD-BASE (T=1000)")

In [ ]:
# RGD with softmax approx for L infinity, X0 from RJD

obj_softmax = lambda L,X : g(alpha, L, X, f)
dobj_softmax = lambda L,X : d_g(alpha, L, X, f, d_f)

results_s1 = []

for _ in tqdm(range(nb_repet)):
    X0_rjd = rjd_base(L,k,2) # generate starting point from RJD
    assert not np.isnan(weights_softmax(alpha, L, X0_rjd).any())
    X_best1, obj_vec1, sG_vec1, iter1, time1 = riemannian_gd(obj_softmax, dobj_softmax, L, X0_rjd, step_gd, nb_iter, tol_conv)  
    results_s1.append(make_result(compute_scores(X_best1, L, k, Y_true), iter1, time1))
add_avgresults_to_df(results_s1, "RGD soft") # softmax X0=RJD(T=2)

print(f"Objective value of 1st iteration: {obj_vec1[0]:.4f}, last iteration: {obj_vec1[-1]:.4f}")
print(f"sG(X) at 1st iteration: {sG_vec1[0]:.4f}, last iteration: {sG_vec1[-1]:.4f}")
fig, ax = plt.subplots(1,2, figsize=(12,5))
ax[0].plot(obj_vec1)
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('Objective value')
ax[0].set_title('RGD Objective Value (softmax)')
ax[1].plot(sG_vec1)
ax[1].set_xlabel('Iteration')
ax[1].set_ylabel('sG(X)')
ax[1].set_title('RGD sG(X)')
plt.suptitle('RGD with softmax approx for L infinity, X0 from RJD')
plt.show()


In [ ]:
# RGD with quasimax approx for L infinity, X0 rjd

# define lambda functions to simplify arguments
obj_quasimax = lambda L,X : h(alpha, L, X, f)
dobj_quasimax = lambda L,X : d_h(alpha, L, X, d_f)

results_q1 = []
    
for _ in tqdm(range(nb_repet)):
    X0_rjd = rjd_base(L,k,2)
    X_best11, obj_vec11, sG_vec11, iter11, time11 = riemannian_gd(obj_quasimax, dobj_quasimax, L, X0_rjd, step_gd, nb_iter, tol_conv)   
    results_q1.append(make_result(compute_scores(X_best11, L, k, Y_true), iter11, time11))
add_avgresults_to_df(results_q1, "RGD quasi") #quasimax X0=RJD(T=2)

print(f"Objective value of 1st iteration: {obj_vec11[0]:.4f}, last iteration: {obj_vec11[-1]:.4f}")
print(f"sG(X) at 1st iteration: {sG_vec11[0]:.4f}, last iteration: {sG_vec11[-1]:.4f}")
fig, ax = plt.subplots(1,2, figsize=(12,5))
ax[0].plot(obj_vec11)
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('Objective value')
ax[0].set_title('RGD Objective Value (quasimax)')
ax[1].plot(sG_vec11)
ax[1].set_xlabel('Iteration')
ax[1].set_ylabel('sG(C)')
ax[1].set_title('RGD sG(C)')
plt.suptitle('RGD with quasimax approx for L infinity, X0 from RJD')
plt.show()

In [ ]:
# RGD with softmax approx for L infinity, X0 random orthogonal matrix

if example!="caltech":
    obj_softmax = lambda L,X : g(alpha, L, X, f)
    dobj_softmax = lambda L,X : d_g(alpha, L, X, f, d_f)

    results_s2 = []
        
    for _ in tqdm(range(nb_repet)):
        X0_random = np.random.rand(n,k)
        X0_random, _ = np.linalg.qr(X0_random)
        assert not np.isnan(weights_softmax(alpha, L, X0_random).any())
        X_best2, obj_vec2, sG_vec2, iter2, time2 = riemannian_gd(obj_softmax, dobj_softmax, L, X0_random, step_gd, nb_iter, tol_conv)  
        results_s2.append(make_result(compute_scores(X_best2, L, k, Y_true), iter2, time2))
    add_avgresults_to_df(results_s2, "RGD soft (random start)")

    print(f"Objective value of 1st iteration: {obj_vec2[0]:.4f}, last iteration: {obj_vec2[-1]:.4f}")
    print(f"sG(X) at 1st iteration: {sG_vec2[0]:.4f}, last iteration: {sG_vec2[-1]:.4f}")
    fig, ax = plt.subplots(1,2, figsize=(12,5))
    ax[0].plot(obj_vec2)
    ax[0].set_xlabel('Iteration')
    ax[0].set_ylabel('Objective value')
    ax[0].set_title('RGD Objective Value (softmax)')
    ax[1].plot(sG_vec2)
    ax[1].set_xlabel('Iteration')
    ax[1].set_ylabel('sG(X)')
    ax[1].set_title('RGD sG(X)')
    plt.suptitle('RGD with softmax approx for L infinity, X0 random')
    plt.show()


In [ ]:
# RGD with quasimax approx for L infinity, X0 random

if example!="caltech":
    obj_quasimax = lambda L,X : h(alpha, L, X, f)
    dobj_quasimax = lambda L,X : d_h(alpha, L, X, d_f)

    results_q2 = []

    for _ in tqdm(range(nb_repet)):
        X0_random = np.random.rand(n,k)
        X0_random, _ = np.linalg.qr(X0_random)
        assert not np.isnan(weights_softmax(alpha, L, X0_random).any())
        X_best21, obj_vec21, sG_vec21, iter21, time21 = riemannian_gd(obj_quasimax, dobj_quasimax, L, X0_random, step_gd, nb_iter, tol_conv)    
        results_q2.append(make_result(compute_scores(X_best21, L, k, Y_true), iter21, time21))
    add_avgresults_to_df(results_q2, "RGD quasi (random start)")

    print(f"Objective value of 1st iteration: {obj_vec21[0]:.4f}, last iteration: {obj_vec21[-1]:.4f}")
    print(f"sG(X) at 1st iteration: {sG_vec21[0]:.4f}, last iteration: {sG_vec21[-1]:.4f}")
    fig, ax = plt.subplots(1,2, figsize=(12,5))
    ax[0].plot(obj_vec21)
    ax[0].set_xlabel('Iteration')
    ax[0].set_ylabel('Objective value')
    ax[0].set_title('RGD Objective Value (quasimax)')
    ax[1].plot(sG_vec21)
    ax[1].set_xlabel('Iteration')
    ax[1].set_ylabel('sG(C)')
    ax[1].set_title('RGD sG(C)')
    plt.suptitle('RGD with quasimax approx for L infinity, X0 random')
    plt.show()

### Comparison results

In [ ]:
df_mean

In [ ]:
filename = 'results__new_tmp2_' + example + '.tex'
# resultdf_to_latex(df_scores, filename, example, nb_repet)
# df_scores
df_combined = mean_ci_results_to_file(df_mean, df_std, df_ci_low, df_ci_upp, ALL_KEYS, confidence_label="95% CI", use_ci=True)
resultdf_to_latex_2tables(df_combined, filename, example, nb_repet, SCORE_KEYS, EXTRA_KEYS, True)
df_combined

## Use subspace generated by 2/3 approx to lower dimension of the problem

In [ ]:
def generate_subspace(L,k):
    X1 = rjd_base(L, k, num_trials=1)
    X2 = rjd_base(L, k, num_trials=1)
    X3 = rjd_base(L, k, num_trials=1)
    Y = np.hstack([X1, X2, X3])
    Y, _ = np.linalg.qr(Y) # orthonormalize Y
    Yt = np.transpose(Y)
    YtY = Yt @ Y
    new_L = Yt@L@Y
    return new_L, Y

In [ ]:
# find C s.t. X0_rjd = Y C
# C0_rjd = np.linalg.lstsq(Y, X0_rjd, rcond=None)[0]
# C0_rjd, _ = np.linalg.qr(C0_rjd) # orthonormalize

In [ ]:
# RGD with softmax approx for L infinity, C0 s.t. X0_rjd = Y C0

obj_softmax = lambda L,X : g(alpha, L, X, f)
dobj_softmax = lambda L,X : d_g(alpha, L, X, f, d_f)
    
results_s5 = []
for _ in tqdm(range(nb_repet)):
    new_L, Y = generate_subspace(L,k)
    C0_rjd = np.linalg.lstsq(Y, X0_rjd, rcond=None)[0]
    C0_rjd, _ = np.linalg.qr(C0_rjd) # orthonormalize
    C_best3, obj_vec5, sG_vec5, iter5, time5 = riemannian_gd(obj_softmax, dobj_softmax, new_L, C0_rjd, step_gd, nb_iter, tol_conv)    
    X_best5 = Y@C_best3
    scores_rgd_softmax_lowerdim2 = dict(zip(SCORE_KEYS, compute_scores(X_best5, L, k, Y_true)))
    results_s5.append(make_result(compute_scores(X_best5, L, k, Y_true), iter5, time5))
add_avgresults_to_df(results_s5, "RGDS soft")

print(f"Objective value of 1st iteration: {obj_vec5[0]:.4f}, last iteration: {obj_vec5[-1]:.4f}")
print(f"sG(X) at 1st iteration: {sG_vec5[0]:.4f}, last iteration: {sG_vec5[-1]:.4f}")
fig, ax = plt.subplots(1,2, figsize=(12,5))
ax[0].plot(obj_vec5)
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('Objective value')
ax[0].set_title('RGD Objective Value (softmax)')
ax[1].plot(sG_vec5)
ax[1].set_xlabel('Iteration')
ax[1].set_ylabel('sG(C)')
ax[1].set_title('RGD sG(C)')
plt.suptitle('RGD with softmax approx for L infinity, C0 from RJD')
plt.show()

In [ ]:
# RGD with quasimax approx for L infinity, C0 s.t. X0_rjd = Y C0

# define lambda functions to simplify arguments
obj_quasimax = lambda L,X : g(alpha, L, X, f)
dobj_quasimax = lambda L,X : d_g(alpha, L, X, f, d_f)

results_q5 = []
for _ in tqdm(range(nb_repet)):
    new_L, Y = generate_subspace(L,k)
    C0_rjd = np.linalg.lstsq(Y, X0_rjd, rcond=None)[0]
    C0_rjd, _ = np.linalg.qr(C0_rjd) # orthonormalize
    C_best4, obj_vec6, sG_vec6, iter6, time6 = riemannian_gd(obj_quasimax, dobj_quasimax, new_L, C0_rjd, step_gd, nb_iter, tol_conv)    
    X_best6 = Y@C_best4
    scores_rgd_quasimax_lowerdim2 = dict(zip(SCORE_KEYS, compute_scores(X_best6, L, k, Y_true)))
    results_q5.append(make_result(compute_scores(X_best6, L, k, Y_true), iter6, time6))
add_avgresults_to_df(results_q5, "RGDS quasi")

print(f"Objective value of 1st iteration: {obj_vec6[0]:.4f}, last iteration: {obj_vec6[-1]:.4f}")
print(f"sG(X) at 1st iteration: {sG_vec6[0]:.4f}, last iteration: {sG_vec6[-1]:.4f}")
fig, ax = plt.subplots(1,2, figsize=(12,5))
ax[0].plot(obj_vec6)
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('Objective value')
ax[0].set_title('RGD Objective Value (quasimax)')
ax[1].plot(sG_vec6)
ax[1].set_xlabel('Iteration')
ax[1].set_ylabel('sG(C)')
ax[1].set_title('RGD sG(C)')
plt.suptitle('RGD with quasimax approx for L infinity, C0 from RJD')
plt.show()

In [ ]:
# RGD with softmax approx for L infinity, C0 random orthogonal matrix
if example!="caltech":
    obj_softmax = lambda L,X : g(alpha, L, X, f)
    dobj_softmax = lambda L,X : d_g(alpha, L, X, f, d_f)

    results_s3 = []
    for _ in tqdm(range(nb_repet)):
        new_L, Y = generate_subspace(L,k)
        C0_random = np.random.rand(3*k,k)
        C0, _ = np.linalg.qr(C0_random)
        C_best1, obj_vec3, sG_vec3, iter3, time3 = riemannian_gd(obj_softmax, dobj_softmax, new_L, C0, step_gd, nb_iter, tol_conv)    
        X_best3 = Y@C_best1
        results_s3.append(make_result(compute_scores(X_best3, L, k, Y_true), iter3, time3))
    add_avgresults_to_df(results_s3, "RGDS soft (random start)")

    print(f"Objective value of 1st iteration: {obj_vec3[0]:.4f}, last iteration: {obj_vec3[-1]:.4f}")
    print(f"sG(X) at 1st iteration: {sG_vec3[0]:.4f}, last iteration: {sG_vec3[-1]:.4f}")
    fig, ax = plt.subplots(1,2, figsize=(12,5))
    ax[0].plot(obj_vec3)
    ax[0].set_xlabel('Iteration')
    ax[0].set_ylabel('Objective value')
    ax[0].set_title('RGD Objective Value (softmax)')
    ax[1].plot(sG_vec3)
    ax[1].set_xlabel('Iteration')
    ax[1].set_ylabel('sG(C)')
    ax[1].set_title('RGD sG(C)')
    plt.suptitle('RGD with softmax approx for L infinity, C0 random')
    plt.show()

In [ ]:
# RGD with quasimax approx for L infinity, C0 random orthogonal matrix

if example!="caltech":
    obj_quasimax = lambda L,X : h(alpha, L, X, f)
    dobj_quasimax = lambda L,X : d_h(alpha, L, X, d_f)

    results_q4 = []
    for _ in tqdm(range(nb_repet)):
        new_L, Y = generate_subspace(L,k)
        C0_random = np.random.rand(3*k,k)
        C0, _ = np.linalg.qr(C0_random)
        C_best2, obj_vec4, sG_vec4, iter4, time4 = riemannian_gd(obj_quasimax, dobj_quasimax, new_L, C0, step_gd, nb_iter, tol_conv)    
        X_best4 = Y@C_best2
        results_q4.append(make_result(compute_scores(X_best4, L, k, Y_true), iter4, time4))
    add_avgresults_to_df(results_q4, "RGDS quasi (random start)")

    print(f"Objective value of 1st iteration: {obj_vec4[0]:.4f}, last iteration: {obj_vec4[-1]:.4f}")
    print(f"sG(X) at 1st iteration: {sG_vec4[0]:.4f}, last iteration: {sG_vec4[-1]:.4f}")
    fig, ax = plt.subplots(1,2, figsize=(12,5))
    ax[0].plot(obj_vec4)
    ax[0].set_xlabel('Iteration')
    ax[0].set_ylabel('Objective value')
    ax[0].set_title('RGD Objective Value (softmax)')
    ax[1].plot(sG_vec4)
    ax[1].set_xlabel('Iteration')
    ax[1].set_ylabel('sG(C)')
    ax[1].set_title('RGD sG(C)')
    plt.suptitle('RGD with softmax approx for L infinity, C0 random')
    plt.show()

## Comparison all methods

In [ ]:
for df in [df_mean, df_std, df_ci_low, df_ci_upp]:
    df["time/iter (ms)"] = df["time/iter (avg)"]*1000
    df.drop(columns=["time/iter (avg)"], inplace=True)

# remove AMI and replace time by ms in keys
ALL_KEYS = ["sG", "subspace invariance", "NMI", "nb iter", "time/iter (ms)"]
SCORE_KEYS = ["sG", "subspace invariance", "NMI"]
EXTRA_KEYS = ["nb iter", "time/iter (ms)"]

In [ ]:
filename = 'results_' + example + '.tex'
df_combined = mean_ci_results_to_file(df_mean, df_std, df_ci_low, df_ci_upp, ALL_KEYS, confidence_label="95% CI", use_ci=True)
resultdf_to_latex_2tables(df_combined, filename, example, nb_repet, SCORE_KEYS, EXTRA_KEYS, True)
df_combine

In [ ]:
# save in pickle file if want to use results in another manner
df_mean.to_pickle(example + '_mean_df')
df_std.to_pickle(example + '_std_df')
df_ci_low.to_pickle(example + '_ci_low_df')
df_ci_upp.to_pickle(example + '_ci_upp_df')